In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
df_geo = pd.read_csv('devices.csv')

df = df.merge(df_geo, on='deviceId', how='left')

In [2]:
df.head()

,deviceId,timedate,period,t1,t2,t3,t4,t5,t6,t7,...,t10,t11,t12,t13,x1,x2,x3,deviceType,latitude,longitude
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:00:00,train,0.29,0.05,0.0,0.43,0.47,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:05:00,train,0.29,0.05,0.0,0.39,0.46,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:10:00,train,0.29,0.05,0.0,0.38,0.46,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:15:00,train,0.29,0.05,0.0,0.38,0.45,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:20:00,train,0.29,0.05,0.0,0.37,0.45,0.45,0.2,...,0.21,0.21,0.07,0.07,0.0,0.0,8,19,50.0,18.3


In [3]:
import numpy as np

features = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10', 't11', 't12', 't13', 'x1', 'x3', 'latitude', 'longitude', 'deviceType']

corr_matrix = df[features].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

threshold = 0.90
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

print(f"{to_drop}")

features_filtered = [f for f in features if f not in to_drop]

['t6', 't12', 't13']


In [5]:
df['timedate'] = pd.to_datetime(df['timedate'])

df['year_month'] = df['timedate'].dt.to_period('D')

# features = features_filtered

agg_funcs = {feat: ['mean', 'std', 'min', 'max'] for feat in features}

agg_funcs['x2'] = ['mean']

df_daily = df.groupby(['deviceId', 'year_month']).agg(agg_funcs).reset_index()

df_daily.columns = ['_'.join(col).strip() if col[1] else col[0] for col in df_daily.columns.values]

df_daily = df_daily.dropna()

df_daily.to_csv('train_daily.csv', index=False)

X = df_daily.drop(columns=['deviceId', 'year_month', 'x2_mean'])
y = df_daily['x2_mean']

In [ ]:
df_daily.columns

Index(['deviceId', 'year_month', 't1_mean', 't1_std', 't1_min', 't1_max',
       't2_mean', 't2_std', 't2_min', 't2_max', 't3_mean', 't3_std', 't3_min',
       't3_max', 't4_mean', 't4_std', 't4_min', 't4_max', 't5_mean', 't5_std',
       't5_min', 't5_max', 't7_mean', 't7_std', 't7_min', 't7_max', 't8_mean',
       't8_std', 't8_min', 't8_max', 't9_mean', 't9_std', 't9_min', 't9_max',
       't10_mean', 't10_std', 't10_min', 't10_max', 't11_mean', 't11_std',
       't11_min', 't11_max', 'x1_mean', 'x1_std', 'x1_min', 'x1_max',
       'x3_mean', 'x3_std', 'x3_min', 'x3_max', 'latitude_mean',
       'latitude_std', 'latitude_min', 'latitude_max', 'longitude_mean',
       'longitude_std', 'longitude_min', 'longitude_max', 'x2_mean'],
      dtype='object')

In [ ]:
import os

scratch_path = os.environ.get('SCRATCH')
ray_tmp_dir = os.path.join(scratch_path, 'ray_tmp')
os.makedirs(ray_tmp_dir, exist_ok=True)
os.environ['RAY_TMPDIR'] = ray_tmp_dir

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor
    
train_data = df_daily.drop(columns=['deviceId', 'year_month'])

train_data = TabularDataset(train_data)

predictor = TabularPredictor(
    label='x2_mean', 
    eval_metric='mean_absolute_error'
).fit(
    train_data,
    time_limit=300,
    presets='best_quality',
    num_gpus=1
)

In [ ]:
import pandas as pd
df = None


In [ ]:

df_valid = pd.read_csv('valid.csv')
df_test = pd.read_csv('test.csv')

df_combined = pd.concat([df_valid, df_test], ignore_index=True)
df_combined = df_combined.merge(df_geo, on='deviceId', how='left')

df_combined['timedate'] = pd.to_datetime(df_combined['timedate'])
df_combined['year'] = df_combined['timedate'].dt.year
df_combined['month'] = df_combined['timedate'].dt.month
df_combined['day'] = df_combined['timedate'].dt.day

features = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 't10', 't11', 't12', 't13', 'x1', 'x3', 'latitude', 'longitude']
agg_funcs = {feat: ['mean', 'std', 'min', 'max'] for feat in features}

df_combined_daily = df_combined.groupby(['deviceId', 'year', 'month', 'day']).agg(agg_funcs).reset_index()

df_combined_daily.columns = [
    '_'.join(col).strip() if col[1] else col[0] 
    for col in df_combined_daily.columns.values
]

X_test = df_combined_daily.drop(columns=['deviceId', 'year', 'month', 'day'])

test_data = TabularDataset(df_combined_daily)
predictions = predictor.predict(test_data)

df_combined_daily['daily_prediction'] = predictions

submission = df_combined_daily.groupby(['deviceId', 'year', 'month'])['daily_prediction'].mean().reset_index()

submission.rename(columns={'daily_prediction': 'prediction'}, inplace=True)

submission.to_csv('submission.csv', index=False)

NameError: name 'predictor' is not defined